# Piper Trainer RU — Qwen3-ASR + Google Drive

Один Colab для подготовки русского датасета и fine-tune Piper.

Что делает:
- монтирует Google Drive;
- использует `Qwen/Qwen3-ASR-1.7B-hf` для автоматической русской расшифровки;
- режет длинные записи по паузам, приводит их к mono 22050 Гц;
- создаёт `metadata.csv`;
- fine-tune Piper от `ru_RU-dmitri-medium` checkpoint;
- сохраняет checkpoints на Google Drive с настраиваемой частотой и лимитом;
- экспортирует `.onnx + .onnx.json` и ZIP для macOS Piper Voice.

> Для голоса реального человека используйте записи, которые вы вправе использовать. Синтетические записи публичных лиц лучше явно обозначать как синтетические.


In [ ]:
#@title 1. Установка зависимостей
import os, sys, subprocess, textwrap, pathlib

def run(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg espeak-ng build-essential cmake ninja-build git")
run("python -m pip install -q -U pip wheel 'setuptools<82' 'jedi>=0.16'")
run("python -m pip install -q 'transformers>=5.13.1' accelerate gradio pydub soundfile librosa huggingface_hub sentencepiece safetensors scikit-build onnx onnxscript")

PIPER_DIR = "/content/piper1-gpl"
PIPER_COMMIT = "5b355b110aecf3de8f4e000ede1ce06831acff35"
if not os.path.exists(PIPER_DIR):
    run("git clone -q https://github.com/OHF-Voice/piper1-gpl.git " + PIPER_DIR)

run(f"cd '{PIPER_DIR}' && git fetch -q origin {PIPER_COMMIT} && git checkout -q {PIPER_COMMIT}")

# Compatibility patch for PyTorch 2.9+ / 2.11:
# Piper's exporter predates the new torch.export-based ONNX path.
_export_py = pathlib.Path(PIPER_DIR) / "src/piper/train/export_onnx.py"
_export_src = _export_py.read_text(encoding="utf-8")
if "dynamo=False" not in _export_src:
    _export_src = _export_src.replace(
        '        dynamic_axes={\n',
        '        dynamo=False,\n        dynamic_axes={\n',
        1,
    )
    _export_py.write_text(_export_src, encoding="utf-8")

run(f"python -m pip install -q -e '{PIPER_DIR}[train]'")
run(f"cd '{PIPER_DIR}' && ./build_monotonic_align.sh")
run(f"cd '{PIPER_DIR}' && python setup.py build_ext --inplace -q")
run(f"cd '{PIPER_DIR}' && python -m piper.train fit --help >/tmp/piper_train_help.txt")
run(f"cd '{PIPER_DIR}' && python -m piper.train.export_onnx --help >/tmp/piper_export_help.txt")

print("Piper training CLI: OK")
print("Piper ONNX export CLI: OK")
print("Piper commit:", PIPER_COMMIT)
print("Готово. Следующая ячейка подключит Google Drive и скачает базовый checkpoint.")


In [ ]:
#@title 2. Google Drive и базовый checkpoint
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from huggingface_hub import hf_hub_download
import os, shutil

ROOT = Path("/content/piper_trainer")
ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/PiperTrainer")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

BASE_DIR = ROOT / "base"
BASE_DIR.mkdir(exist_ok=True)

DRIVE_BASE = DRIVE_ROOT / "_base"
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

def cached_base_file(filename):
    drive_file = DRIVE_BASE / filename
    if not drive_file.exists():
        downloaded = hf_hub_download(
            repo_id="rhasspy/piper-checkpoints",
            repo_type="dataset",
            filename="ru/ru_RU/dmitri/medium/" + filename,
            local_dir=str(DRIVE_BASE / "_download"),
        )
        shutil.copy2(downloaded, drive_file)

    local_file = BASE_DIR / filename
    if (
        not local_file.exists()
        or local_file.stat().st_size != drive_file.stat().st_size
    ):
        shutil.copy2(drive_file, local_file)
    return str(local_file)

DMITRI_CKPT = cached_base_file("epoch=5589-step=1478840.ckpt")
DMITRI_CONFIG = cached_base_file("config.json")

print("Базовый checkpoint:", DMITRI_CKPT)
print("Постоянный кэш базы:", DRIVE_BASE)
print("Google Drive:", DRIVE_ROOT)


In [ ]:
#@title 3. Backend: подготовка датасета, ASR и хранение
import os, gc, csv, json, math, shutil, zipfile, subprocess, threading, time, re
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from transformers import AutoProcessor, AutoModelForMultimodalLM

WORK_ROOT = Path("/content/piper_trainer")
ASR_MODEL = "Qwen/Qwen3-ASR-1.7B-hf"
_asr = None
_train_process = None

def environment_status():
    parts = []
    parts.append(f"Python: {sys.version.split()[0]}")
    parts.append(f"PyTorch: {torch.__version__}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram = props.total_memory / (1024 ** 3)
        parts.append(f"GPU: {torch.cuda.get_device_name(0)}")
        parts.append(f"VRAM: {vram:.1f} ГБ")
        parts.append("CUDA: доступна")
    else:
        parts.append("GPU: НЕ НАЙДЕН")
        parts.append("Откройте Runtime → Change runtime type → T4 GPU")
    parts.append(
        "Google Drive: подключён"
        if DRIVE_ROOT.exists()
        else "Google Drive: не подключён"
    )
    parts.append(f"Qwen ASR: {ASR_MODEL}")
    parts.append(f"Базовый Piper checkpoint: {Path(DMITRI_CKPT).name}")
    return "\n".join(parts)

def safe_name(name: str) -> str:
    name = re.sub(r"[^0-9A-Za-zА-Яа-яЁё_-]+", "_", name.strip())
    return name.strip("_") or "voice"

def project_paths(project: str):
    project = safe_name(project)
    local = WORK_ROOT / project
    drive_p = DRIVE_ROOT / project
    audio = local / "dataset" / "wav"
    audio.mkdir(parents=True, exist_ok=True)
    drive_p.mkdir(parents=True, exist_ok=True)
    return local, drive_p, audio

def load_asr():
    global _asr
    if _asr is None:
        if not torch.cuda.is_available():
            raise RuntimeError("GPU не найден. В Colab выберите Runtime → Change runtime type → T4 GPU.")
        print("Загружаю", ASR_MODEL)
        processor = AutoProcessor.from_pretrained(ASR_MODEL)
        model = AutoModelForMultimodalLM.from_pretrained(
            ASR_MODEL,
            dtype=torch.float16,
            device_map="auto",
        )
        model.eval()
        _asr = (processor, model)
    return _asr

def transcribe_file(path):
    processor, model = load_asr()
    inputs = processor.apply_transcription_request(
        audio=str(path),
        language="ru",
    ).to(model.device, model.dtype)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=512)

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated_ids,
        return_format="transcription_only",
    )
    if isinstance(text, (list, tuple)):
        if len(text) == 1:
            text = text[0]
        else:
            text = " ".join(str(part) for part in text)
    return normalize_text(text)

def test_qwen_asr(file_path):
    if not file_path:
        raise ValueError("Выберите один аудиофайл для проверки Qwen.")
    text = transcribe_file(file_path)
    return "Qwen3-ASR распознал:\n" + text

def unload_asr():
    global _asr
    _asr = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


ALIGNER_MODEL = "Qwen/Qwen3-ForcedAligner-0.6B-hf"
_aligner = None

def load_aligner():
    """Load only after ASR has been unloaded; T4 cannot safely hold both."""
    global _aligner
    if _aligner is None:
        if not torch.cuda.is_available():
            raise RuntimeError("Для привязки слов к записи требуется GPU Colab.")
        from transformers import AutoModelForTokenClassification
        processor = AutoProcessor.from_pretrained(ALIGNER_MODEL)
        model = AutoModelForTokenClassification.from_pretrained(
            ALIGNER_MODEL, dtype=torch.float16, device_map="auto"
        )
        model.eval()
        _aligner = (processor, model)
    return _aligner

def align_context(audio_path, transcript):
    """Align the existing ASR transcript. Does NOT recognize the audio again."""
    processor, model = load_aligner()
    inputs, word_lists = processor.prepare_forced_aligner_inputs(
        audio=str(audio_path), transcript=transcript, language="Russian"
    )
    inputs = inputs.to(model.device, model.dtype)
    with torch.inference_mode():
        outputs = model(**inputs)
    words = processor.decode_forced_alignment(
        logits=outputs.logits,
        input_ids=inputs["input_ids"],
        word_lists=word_lists,
        timestamp_token_id=model.config.timestamp_token_id,
    )[0]
    return words

def unload_aligner():
    global _aligner
    _aligner = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def normalize_clip_loudness(clip, target_dbfs=-20.0, peak_ceiling_dbfs=-1.0):
    """Normalize one speech clip while keeping at least 1 dB of peak headroom."""
    if len(clip) == 0 or clip.rms == 0:
        return clip
    gain = float(target_dbfs) - float(clip.dBFS)
    clip = clip.apply_gain(gain)
    if clip.max_dBFS > float(peak_ceiling_dbfs):
        clip = clip.apply_gain(float(peak_ceiling_dbfs) - float(clip.max_dBFS))
    return clip

def phrase_boundary_reason(audio, start_ms, end_ms, padding_ms):
    clip = audio[int(start_ms):int(end_ms)]
    if len(clip) < 500 or clip.rms == 0:
        return None
    edge_ms = min(180, max(80, int(padding_ms) // 2))
    body_db = float(clip.dBFS)
    if int(start_ms) > 0:
        left = clip[:edge_ms]
        if left.rms and float(left.dBFS) > body_db - 8.0:
            return "начало фразы обрезано не по тихой паузе"
    if int(end_ms) < len(audio):
        right = clip[-edge_ms:]
        if right.rms and float(right.dBFS) > body_db - 8.0:
            return "конец фразы обрезан не по тихой паузе"
    return None

def _quietest_cut_ms(audio, start_ms, end_ms, min_piece_ms, max_piece_ms):
    """Choose a low-energy cut instead of blindly cutting at max_piece_ms."""
    lo = start_ms + max(int(min_piece_ms), int(max_piece_ms * 0.55))
    hi = min(start_ms + int(max_piece_ms), end_ms - int(min_piece_ms))
    if hi <= lo:
        return min(start_ms + int(max_piece_ms), end_ms)

    target = start_ms + int(max_piece_ms * 0.82)
    best = None
    # 80 ms window every 20 ms is enough to find inter-word valleys.
    for cut in range(lo, hi + 1, 20):
        window = audio[max(start_ms, cut - 40):min(end_ms, cut + 40)]
        db = -100.0 if len(window) == 0 or window.rms == 0 else float(window.dBFS)
        distance_penalty = abs(cut - target) / max_piece_ms * 4.0
        score = db + distance_penalty
        if best is None or score < best[0]:
            best = (score, cut)
    return best[1] if best else min(start_ms + int(max_piece_ms), end_ms)

def _split_long_nonsilent_range(audio, start_ms, end_ms, min_piece_ms, max_piece_ms):
    """Split continuous speech only at the quietest available point."""
    out = []
    cur = int(start_ms)
    end_ms = int(end_ms)
    while end_ms - cur > int(max_piece_ms):
        cut = _quietest_cut_ms(
            audio,
            cur,
            end_ms,
            min_piece_ms=min_piece_ms,
            max_piece_ms=max_piece_ms,
        )
        if cut <= cur:
            cut = min(cur + int(max_piece_ms), end_ms)
        out.append((cur, cut))
        cur = cut
    if end_ms > cur:
        out.append((cur, end_ms))
    return out

def smart_segment_ranges(
    audio,
    min_silence_ms=250,
    silence_db=-35.0,
    min_sec=1.5,
    max_sec=9.0,
    padding_ms=350,
    merge_gap_ms=650,
):
    """Create TTS-friendly phrases using real pauses instead of fixed-time chopping."""
    min_ms = int(float(min_sec) * 1000)
    max_ms = int(float(max_sec) * 1000)

    raw = detect_nonsilent(
        audio,
        min_silence_len=int(min_silence_ms),
        silence_thresh=float(silence_db),
        seek_step=10,
    )
    if not raw:
        return []

    # Pack neighboring speech islands only while the whole phrase stays under max_ms.
    packed = []
    current = None
    for start, end in raw:
        start, end = int(start), int(end)
        if end - start > max_ms:
            if current is not None:
                packed.append(tuple(current))
                current = None
            packed.extend(
                _split_long_nonsilent_range(
                    audio,
                    start,
                    end,
                    min_piece_ms=min_ms,
                    max_piece_ms=max_ms,
                )
            )
            continue

        if current is None:
            current = [start, end]
            continue

        gap = start - current[1]
        if gap <= int(merge_gap_ms) and end - current[0] <= max_ms:
            current[1] = end
        else:
            packed.append(tuple(current))
            current = [start, end]

    if current is not None:
        packed.append(tuple(current))

    # Absorb very short phrases into a neighbor when that still respects max_ms.
    packed = [list(x) for x in packed]
    i = 0
    while i < len(packed):
        duration = packed[i][1] - packed[i][0]
        if duration < min_ms and len(packed) > 1:
            if (
                i + 1 < len(packed)
                and packed[i + 1][0] - packed[i][1] <= int(merge_gap_ms)
                and packed[i + 1][1] - packed[i][0] <= max_ms
            ):
                packed[i + 1][0] = packed[i][0]
                packed.pop(i)
                continue
            if (
                i > 0
                and packed[i][0] - packed[i - 1][1] <= int(merge_gap_ms)
                and packed[i][1] - packed[i - 1][0] <= max_ms
            ):
                packed[i - 1][1] = packed[i][1]
                packed.pop(i)
                i -= 1
                continue
        i += 1

    # Give the silence between neighboring utterances to each side once.
    # Padding both pieces by 350 ms used to duplicate words in adjacent WAVs.
    final = []
    for i, (start, end) in enumerate(packed):
        left_limit = 0 if i == 0 else (int(packed[i - 1][1]) + int(start)) // 2
        right_limit = (
            len(audio) if i + 1 == len(packed)
            else (int(end) + int(packed[i + 1][0])) // 2
        )
        padded_start = max(left_limit, int(start) - int(padding_ms))
        padded_end = min(right_limit, int(end) + int(padding_ms))
        if final:
            padded_start = max(padded_start, final[-1][1])
        if padded_end - padded_start >= min_ms:
            final.append((padded_start, padded_end))
    return final


def natural_clause_boundary(audio, start_ms, end_ms):
    """A mid-sentence utterance can still be valid when both ends are genuine pauses."""
    clip = audio[int(start_ms):int(end_ms)]
    if len(clip) < 1200 or clip.rms == 0:
        return False
    body_db = float(clip.dBFS)
    for border, edge in ((int(start_ms), clip[:150]), (int(end_ms), clip[-150:])):
        if border in (0, len(audio)):
            continue
        if edge.rms and float(edge.dBFS) > body_db - 8.0:
            return False
    return True

def unfinished_syntax(text):
    """Block clear dangling function words even when ASR omits final punctuation."""
    text = normalize_text(text)
    if text.endswith(("...", ",", ":", ";", "—", "-")):
        return True
    words = re.findall(r"[А-Яа-яЁёA-Za-z]+", text.lower())
    if len(words) < 2:
        return True
    if len(words) < 4 and not text.endswith((".", "!", "?")):
        return True
    dangling = {
        "а", "и", "но", "или", "если", "когда", "потому", "хотя", "однако",
        "что", "чтобы", "который", "которая", "которое", "которые", "которых",
        "которому", "которой", "которыми", "как", "где", "пока", "ибо",
        "в", "во", "на", "к", "ко", "с", "со", "по", "из", "из-за", "за",
        "до", "от", "о", "об", "обо", "для", "без", "при", "под", "над",
        "через", "между", "перед", "после", "про", "у", "не", "ни", "бы",
        "же", "ли", "это", "этот", "эти", "эту", "то", "тем", "так",
    }
    return words[-1] in dangling

def phrase_quality_reason(text, duration, allow_unpunctuated_clause=False):
    text = normalize_text(text)
    letters = len(re.findall(r"[A-Za-zА-Яа-яЁё]", text))
    cps = len(text) / max(float(duration), 0.001)
    if not text:
        return "пустая расшифровка"
    if letters < 8:
        return "слишком короткая фраза"
    if cps < 3.0:
        return f"слишком мало текста для {duration:.1f} сек"
    if cps > 29.0:
        return f"слишком много текста для {duration:.1f} сек"
    if text[-1] not in ".!?" and not (
        allow_unpunctuated_clause and not unfinished_syntax(text)
    ):
        return "фраза оборвана: нет конечного знака"
    if text.endswith("..."):
        return "фраза звучит незаконченной"
    # Qwen sometimes puts a full stop after an acoustically cut-off "и", "но".
    if unfinished_syntax(text):
        return "обрыв после союза/предлога; прослушайте фрагмент"
    low = text.lower()
    if re.search(r"\b(и и и|это это это|что что что|ну ну ну)\b", low):
        return "подозрительное повторение"
    if text.startswith(("['", '[\"')) or text.endswith(("']", '\"]')):
        return "служебные скобки ASR"
    return None

def normalize_text(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = text.replace("…", "...")
    return text

def validate_dataset_for_training(dataset):
    import wave
    metadata = Path(dataset) / "metadata.csv"
    if not metadata.exists():
        raise FileNotFoundError("metadata.csv не найден. Сначала подготовьте датасет.")
    allowed_path = Path(dataset) / "natural_clause_files.txt"
    natural_clause_files = (
        set(allowed_path.read_text(encoding="utf-8").splitlines())
        if allowed_path.exists() else set()
    )
    problems = []
    phrases = 0
    total_sec = 0.0
    for line_no, raw in enumerate(metadata.read_text(encoding="utf-8").splitlines(), start=1):
        if not raw.strip():
            continue
        if "|" not in raw:
            problems.append(f"строка {line_no}: нет разделителя |")
            continue
        filename, text = raw.split("|", 1)
        wav_path = Path(dataset) / "wav" / filename.strip()
        text = normalize_text(text)
        if not wav_path.exists():
            problems.append(f"строка {line_no}: нет аудио {wav_path.name}")
            continue
        with wave.open(str(wav_path), "rb") as wf:
            duration = wf.getnframes() / max(wf.getframerate(), 1)
        allow_clause = filename.strip() in natural_clause_files
        reason = phrase_quality_reason(
            text, duration, allow_unpunctuated_clause=allow_clause
        )
        if reason:
            problems.append(f"строка {line_no}: {reason}")
            continue
        phrases += 1
        total_sec += duration
    # Two neighboring training WAVs must not contain the same spoken words.
    review = Path(dataset) / "review.csv"
    if review.exists():
        table = pd.read_csv(review)
        if {"file", "start_sec", "end_sec"}.issubset(table.columns):
            table = table.sort_values(["start_sec", "end_sec"])
            prior_end, prior_name = None, None
            for row in table.itertuples(index=False):
                current_start = float(row.start_sec)
                current_end = float(row.end_sec)
                if prior_end is not None and current_start < prior_end - 0.12:
                    problems.append(
                        f"перекрытие речи между {prior_name} и {row.file}: "
                        f"{prior_end - current_start:.2f} сек"
                    )
                if prior_end is None or current_end > prior_end:
                    prior_end, prior_name = current_end, row.file
    if problems:
        preview = "\n".join(problems[:20])
        raise ValueError("Обучение не запущено: найдены сомнительные фразы.\n" + preview)
    if phrases == 0:
        raise ValueError("Обучение не запущено: в metadata.csv нет пригодных фраз.")
    return phrases, total_sec / 60.0


In [ ]:
AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".mp4"}

def expand_input_files(files, local):
    if not isinstance(files, (list, tuple)):
        files = [files]
    imported = local / "_imports"
    if imported.exists():
        shutil.rmtree(imported)
    imported.mkdir(parents=True, exist_ok=True)

    out = []
    serial = 0
    for file_obj in files:
        src = Path(file_obj)
        if src.suffix.lower() != ".zip":
            if src.suffix.lower() in AUDIO_EXTS:
                out.append(src)
            continue

        with zipfile.ZipFile(src) as z:
            for info in z.infolist():
                if info.is_dir():
                    continue
                suffix = Path(info.filename).suffix.lower()
                if suffix not in AUDIO_EXTS:
                    continue
                serial += 1
                target = imported / f"zip_{serial:05d}{suffix}"
                with z.open(info) as source, target.open("wb") as dest:
                    shutil.copyfileobj(source, dest)
                out.append(target)
    return out


def context_blocks(audio, target_ms=24000, min_ms=15000, max_ms=29000,
                   silence_thresh=-35, min_silence_len=190):
    """ASR gets long, sequential blocks of the whole recording, never 3s scraps."""
    from pydub.silence import detect_silence
    blocks, start = [], 0
    while start < len(audio):
        if len(audio) - start <= max_ms:
            blocks.append((start, len(audio), True))
            break
        lo = start + min_ms
        hi = min(len(audio), start + max_ms)
        pauses = detect_silence(
            audio[lo:hi], min_silence_len=190,
            silence_thresh=-35, seek_step=20
        )
        if pauses:
            midpoints = [lo+(a+b)//2 for a,b in pauses]
            boundary = min(midpoints, key=lambda t: abs(t-start-target_ms))
            reliable = True
        else:
            boundary = _quietest_cut_ms(
                audio, start, hi, min_piece_ms=min_ms,
                max_piece_ms=max_ms
            )
            reliable = False
        if boundary < start+min_ms or boundary > hi:
            raise ValueError("Не удалось найти подходящую границу длинного блока.")
        blocks.append((start, boundary, reliable))
        start = boundary
    return blocks

def word_groups(aligned_words, block_seconds, target_sec=4.5,
                min_sec=3.0, max_sec=6.0):
    """Build non-overlapping 3-6s phrases between *words*, not inside them."""
    words=[]
    prev=-1.0
    for item in aligned_words:
        text=str(item["text"]).strip()
        begin, end=float(item["start_time"]),float(item["end_time"])
        if (not text or not math.isfinite(begin) or not math.isfinite(end)
            or begin < -0.04 or end > block_seconds+0.15
            # The aligner often assigns a zero-length point to short
            # function words / punctuation, and sometimes a multi-second
            # span to a word followed by silence. Neither means misalignment.
            or end < begin-0.08 or end-begin > 6.0
            or begin < prev-0.15):
            raise ValueError("Привязка слов не прошла проверку временных меток.")
        begin=max(0.0,min(block_seconds,begin))
        end=max(begin,min(block_seconds,end))
        # Keep the transcription order for small ASR timing jitter; never
        # invent extra words or shift a timestamp by whole seconds.
        begin=max(begin,prev)
        end=max(begin,end)
        words.append({"text":text,"start_time":begin,"end_time":end})
        prev=end
    if not words:
        raise ValueError("Модель не вернула ни одного слова.")

    # Dynamic programming: do not leave a 1-second tail by greedily taking
    # the first 4.5-second group. Every accepted word belongs to one phrase.
    total=len(words)
    best=[None]*(total+1)
    best[total]=(0.0,[])
    for first in range(total-1,-1,-1):
        options=[]
        for last in range(first,total):
            duration=words[last]["end_time"]-words[first]["start_time"]
            if duration>max_sec+0.04:
                break
            if duration<min_sec-0.04 or best[last+1] is None:
                continue
            gap=(words[last+1]["start_time"]-words[last]["end_time"]
                 if last+1<total else 0.0)
            punctuation=bool(re.search(r"[.!?]$",words[last]["text"]))
            score=(duration-target_sec)**2
            score-=min(max(gap,0.0),0.5)*0.55
            if punctuation:
                score-=0.15
            # The alignment can be correct but the resulting phrase can
            # still end on "и", "что", "в" etc. Prefer a nearby word boundary
            # that keeps the following function word with its next phrase.
            # This only changes ranking among valid, non-overlapping 3-6s
            # groupings; it never cuts through a word.
            if last+1<total and unfinished_syntax(
                group_transcript_text(words[first:last+1])
            ):
                score+=8.0
            next_cost, next_ranges=best[last+1]
            options.append((score+next_cost,[(first,last+1)]+next_ranges))
        if options:
            best[first]=min(options,key=lambda x:x[0])
    if best[0] is None:
        raise ValueError(
            "Нет надёжного разбиения на фразы по 3–6 секунд без резки слов."
        )
    return [words[a:b] for a,b in best[0][1]]


def word_group_audio_ranges(groups, block_ms, pad_ms=90):
    """Split inter-word gaps at their midpoint; padding cannot duplicate audio."""
    segments=[]
    for index,group in enumerate(groups):
        first=round(group[0]["start_time"]*1000)
        last=round(group[-1]["end_time"]*1000)
        left=0 if index==0 else round(
            (groups[index-1][-1]["end_time"]+group[0]["start_time"])*500
        )
        right=block_ms if index+1==len(groups) else round(
            (group[-1]["end_time"]+groups[index+1][0]["start_time"])*500
        )
        begin=max(left,first-pad_ms)
        end=min(right,last+pad_ms)
        if segments:
            begin=max(begin,segments[-1][1])
        if end<=begin:
            raise ValueError("Перекрывающиеся или нулевые интервалы.")
        segments.append((begin,end))
    return segments

def alignment_matches_transcript(words, transcript):
    """Reject an alignment that substitutes words, not merely one with fewer.

    A count-only check accepted completely unrelated text of equal length.
    Qwen may change punctuation or hyphens while aligning, so compare
    normalized alphanumeric character streams in order; permit modest
    tokenization noise, but not wholesale word substitution.
    """
    from difflib import SequenceMatcher
    def canonical(value):
        return "".join(
            ch.casefold().replace("ё", "е")
            for ch in str(value) if ch.isalnum()
        )
    spoken = canonical(transcript)
    aligned = canonical(" ".join(str(w["text"]) for w in words))
    if not spoken or not aligned:
        return False
    ratio = min(len(spoken), len(aligned)) / max(len(spoken), len(aligned))
    if ratio < 0.83:
        return False
    return SequenceMatcher(None, spoken, aligned, autojunk=False).ratio() >= 0.86

def group_transcript_text(group):
    text=" ".join(word["text"] for word in group)
    text=re.sub(r"\s+([.,!?;:])",r"\1",text)
    return normalize_text(text)

def punctuate_aligned_groups(groups, transcript):
    """Restore *existing* ASR punctuation; do not change words or invent stops.

    The ForcedAligner returns a simplified word list without commas, periods,
    and many hyphens. Piper must train with the actual long-block ASR text,
    rather than losing all punctuation in every training label.
    """
    original=str(transcript)
    keep=[i for i,c in enumerate(original) if c.isalnum()]
    flatten=lambda value: "".join(
        c.casefold().replace("ё","е") for c in value if c.isalnum()
    )
    normalized=flatten(original)
    cursor=0
    result=[]
    for group in groups:
        bare=group_transcript_text(group)
        token=flatten(bare)
        index=normalized.find(token,cursor) if token else -1
        if index<0 or index+len(token)>len(keep):
            result.append(bare)
            continue
        first,last=keep[index],keep[index+len(token)-1]
        text=original[first:last+1]
        # Attach a final punctuation mark only if the original ASR placed
        # that mark directly after this final word (before the next word).
        tail=re.match(r"\s*([.!?]+)(?=\s|$)",original[last+1:])
        if tail:
            text+=tail.group(1)
        if flatten(text)!=token:
            result.append(bare)
            continue
        result.append(normalize_text(text))
        cursor=index+len(token)
    return result


def prepare_dataset(files, project, min_silence_ms, silence_db, min_sec, max_sec, padding_ms, drive_audio_path="", status=None):
    selected_files = []
    if files:
        selected_files.extend(files if isinstance(files, (list, tuple)) else [files])
    if str(drive_audio_path).strip():
        selected_files.append(str(drive_audio_path).strip())
    files = selected_files
    if _train_process is not None and _train_process.poll() is None:
        raise RuntimeError(
            "Сейчас идёт обучение. Нельзя перестраивать датасет, пока Piper читает его. "
            "Дождитесь завершения или остановите обучение."
        )
    if not files:
        raise ValueError("Загрузите аудио или укажите путь к нему на Google Drive.")
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)

    # Preserve the original recording: a Gradio upload disappears with Colab.
    # Store it outside dataset/ so rebuilding does not erase the only source.
    import hashlib
    source_dir = drive_p / "source"
    source_dir.mkdir(parents=True, exist_ok=True)
    source_manifest = []
    for item in files:
        original = Path(item)
        if not original.is_file():
            raise FileNotFoundError(f"Исходный файл не найден: {original}")
        if source_dir in original.parents:
            source_manifest.append(str(original))
            continue
        digest = hashlib.sha256()
        with original.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        saved = source_dir / f"source_{digest.hexdigest()[:16]}{original.suffix.lower()}"
        if not saved.exists() or saved.stat().st_size != original.stat().st_size:
            shutil.copy2(original, saved)
        source_manifest.append(str(saved))
    (source_dir / "source_paths.txt").write_text(
        "\n".join(source_manifest) + "\n", encoding="utf-8"
    )

    # Build in a separate directory; never delete the previous training set
    # before the candidate has passed ASR and has been copied to Drive.
    dataset_dir = local / "dataset_candidate"
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    wav_dir = dataset_dir / "wav"
    wav_dir.mkdir(parents=True, exist_ok=True)
    files = expand_input_files(files, local)
    if not files:
        raise ValueError("В загрузке нет поддерживаемых аудиофайлов.")

    # PHASE 1: transcribe every long block of every original recording.
    # No 3-6s training WAV is created until the complete source is transcribed.
    records=[]
    context_dir=dataset_dir/"context_audio"
    context_dir.mkdir(parents=True,exist_ok=True)
    full_transcript=[]
    try:
        load_asr()
        for file_index,file_obj in enumerate(files,1):
            src=Path(file_obj)
            audio=AudioSegment.from_file(src).set_channels(1).set_frame_rate(22050)
            segments=context_blocks(
                audio, silence_thresh=silence_db, min_silence_len=min_silence_ms
            )
            for block_index,(begin,end,safe_boundary) in enumerate(segments,1):
                audio_path=context_dir/f"source{file_index:03d}_block{block_index:04d}.wav"
                audio[begin:end].export(
                    audio_path,format="wav",parameters=["-acodec","pcm_s16le"]
                )
                transcript=normalize_text(transcribe_file(audio_path))
                record={"source":src.name,"source_index":file_index,
                        "block_index":block_index,"start_ms":begin,"end_ms":end,
                        "safe_boundary":safe_boundary,
                        "safe_start":block_index==1 or segments[block_index-2][2],
                        "audio_path":str(audio_path),
                        "text":transcript}
                records.append(record)
                full_transcript.append(
                    f"[{src.name} {begin/1000:.2f}–{end/1000:.2f} с] {transcript}"
                )
                print(f"ASR {src.name} {block_index}/{len(segments)}: {transcript[:115]}",flush=True)
            del audio
    finally:
        unload_asr()
    (dataset_dir/"full_transcript.txt").write_text(
        "\n".join(full_transcript)+"\n",encoding="utf-8"
    )
    (dataset_dir/"context_blocks.json").write_text(
        json.dumps([{k:v for k,v in record.items() if k!="audio_path"}
                    for record in records],ensure_ascii=False,indent=2),
        encoding="utf-8"
    )
    # PHASE 2: align every already-recognized long block, and only then
    # split its *word timings* into short, non-overlapping Piper clips.
    rows=[]
    rejected=[]
    natural_clause_files=[]
    counter=0
    review_wav_dir=dataset_dir/"needs_review_wav"
    review_wav_dir.mkdir(parents=True,exist_ok=True)
    try:
        load_aligner()
        for record in records:
            context=AudioSegment.from_wav(record["audio_path"])
            block_ms=len(context)
            src=record["source"]
            base=record["start_ms"]
            def save_rejected(reason,part=None):
                nonlocal counter
                counter+=1
                name=f"{project}_{counter:06d}.wav"
                clip=context if part is None else context[part[0]:part[1]]
                clip.export(review_wav_dir/name,format="wav")
                rejected.append({
                    "file":name,"reason":str(reason),"duration":round(len(clip)/1000,3),
                    "text":record["text"],"source":src,
                    "start_sec":round((base+(0 if part is None else part[0]))/1000,3),
                    "end_sec":round((base+(block_ms if part is None else part[1]))/1000,3),
                    "review_audio":f"needs_review_wav/{name}"
                })
            if not record["text"]:
                save_rejected("Qwen вернула пустую расшифровку длинного блока")
                continue
            try:
                aligned=align_context(record["audio_path"],record["text"])
                words=[{"text":str(word["text"]),"start_time":float(word["start_time"]),
                        "end_time":float(word["end_time"])} for word in aligned]
                if not alignment_matches_transcript(words,record["text"]):
                    raise ValueError("Привязка слов не совпала с текстом расшифровки")
                groups=word_groups(
                    words,block_ms/1000,target_sec=4.5,
                    min_sec=max(2.8,float(min_sec)),
                    max_sec=min(6.0,float(max_sec))
                )
                spans=word_group_audio_ranges(
                    groups,block_ms,pad_ms=min(120,max(35,int(padding_ms)))
                )
                group_texts=punctuate_aligned_groups(groups,record["text"])
            except (ValueError,RuntimeError,IndexError,KeyError) as exc:
                save_rejected(f"Ошибка привязки длинного блока: {exc}")
                continue
            for group,(begin,end),text in zip(groups,spans,group_texts):
                length_sec=(end-begin)/1000
                # A context block forcibly split in the middle of speech is
                # not a trustworthy sentence boundary at its outer edge.
                uncertain_edge=(
                    (not record["safe_boundary"] and end>=block_ms-250)
                    or (not record["safe_start"] and begin<=250)
                )
                reason=None
                if len(group)<2:
                    reason="Слишком мало слов в фразе"
                elif not 2.8<=length_sec<=7.1:
                    reason=f"Неподходящая длительность {length_sec:.2f} с"
                elif uncertain_edge:
                    reason="Длинный блок закончился без подтверждённой паузы"
                elif unfinished_syntax(text):
                    reason="В конце фразы союз или предлог"
                else:
                    characters=len(text)
                    cps=characters/max(length_sec,0.001)
                    if not 3<=cps<=29:
                        reason=f"Подозрительная скорость речи {cps:.1f} символов/сек"
                counter+=1
                name=f"{project}_{counter:06d}.wav"
                clip=normalize_clip_loudness(context[begin:end])
                # The aligner supplies a boundary between words. A loud
                # phoneme near a boundary is not by itself evidence of a cut.
                # Do not throw away valid connected speech due to punctuation
                # or an overly strict loud-edge detector.
                if reason:
                    clip.export(review_wav_dir/name,format="wav")
                    rejected.append({
                        "file":name,"reason":reason,"duration":round(length_sec,3),
                        "text":text,"source":src,
                        "start_sec":round((base+begin)/1000,3),
                        "end_sec":round((base+end)/1000,3),
                        "review_audio":f"needs_review_wav/{name}"
                    })
                    continue
                clip.export(wav_dir/name,format="wav",parameters=["-acodec","pcm_s16le"])
                # A natural mid-sentence pause is valid; punctuation is not
                # a prerequisite when words and timestamps are aligned.
                natural_clause_files.append(name)
                rows.append({
                    "file":name,"text":text,"duration":round(length_sec,3),
                    "source":src,"start_sec":round((base+begin)/1000,3),
                    "end_sec":round((base+end)/1000,3),
                    "chars_per_sec":round(len(text)/length_sec,3),
                    "rms_dbfs":round(float(clip.dBFS),3),
                    "peak_dbfs":round(float(clip.max_dBFS),3)
                })
    finally:
        unload_aligner()
    # Keep the complete ASR text and alignment audit; context WAVs are
    # temporary intermediates, the original source is cached separately.
    if context_dir.exists():
        shutil.rmtree(context_dir)
    recovered_clauses=sum(
        not row["text"].endswith((".","!","?")) for row in rows
    )
    if not rows:
        raise RuntimeError("Не получилось получить ни одной пригодной фразы. Попробуйте снизить порог тишины.")

    df = pd.DataFrame(rows)
    review_csv = dataset_dir / "review.csv"
    df.to_csv(review_csv, index=False, encoding="utf-8")

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for row in rows:
            f.write(f"{row['file']}|{row['text']}\n")

    (dataset_dir / "natural_clause_files.txt").write_text(
        "\n".join(natural_clause_files) + ("\n" if natural_clause_files else ""),
        encoding="utf-8",
    )

    report = {
        "pipeline": "ASR full source in long blocks, align words, split 3-6 s",
        "context_blocks": len(records),
        "full_transcript": "full_transcript.txt",
        "forced_aligner": ALIGNER_MODEL,
        "phrases": int(len(rows)),
        "rejected": int(len(rejected)),
        "recovered_natural_clauses": int(recovered_clauses),
        "review_minutes": round(sum(r["duration"] for r in rejected) / 60.0, 3),
        "minutes": round(float(df["duration"].sum()) / 60.0, 3),
        "duration_min": round(float(df["duration"].min()), 3),
        "duration_median": round(float(df["duration"].median()), 3),
        "duration_max": round(float(df["duration"].max()), 3),
        "rms_dbfs_min": round(float(df["rms_dbfs"].min()), 3),
        "rms_dbfs_median": round(float(df["rms_dbfs"].median()), 3),
        "rms_dbfs_max": round(float(df["rms_dbfs"].max()), 3),
        "longer_than_10_sec": int((df["duration"] > 10).sum()),
        "shorter_than_2_sec": int((df["duration"] < 2).sum()),
        "settings": {
            "min_silence_ms": int(min_silence_ms),
            "silence_db": float(silence_db),
            "min_sec": float(min_sec),
            "max_sec": float(max_sec),
            "padding_ms": int(padding_ms),
            "target_dbfs": -20.0,
            "peak_ceiling_dbfs": -1.0,
        },
    }
    (dataset_dir / "dataset_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    if rejected:
        review_df = pd.DataFrame(rejected)
        review_df.to_csv(dataset_dir / "rejected.csv", index=False, encoding="utf-8")
        review_df.to_csv(dataset_dir / "needs_review.csv", index=False, encoding="utf-8")

    # Stage the complete candidate before replacing an existing dataset.
    drive_dataset = drive_p / "dataset"
    drive_candidate = drive_p / "dataset_candidate"
    if drive_candidate.exists():
        shutil.rmtree(drive_candidate)
    shutil.copytree(dataset_dir, drive_candidate)
    if drive_dataset.exists():
        backup = drive_p / (
            "dataset_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
        )
        shutil.copytree(drive_dataset, backup)
        shutil.rmtree(drive_dataset)
    shutil.move(str(drive_candidate), str(drive_dataset))

    current_local = local / "dataset"
    if current_local.exists():
        shutil.rmtree(current_local)
    shutil.move(str(dataset_dir), str(current_local))
    review_csv = current_local / "review.csv"

    summary = (
        f"Готово: {len(rows)} фраз, "
        f"{df['duration'].sum()/60:.1f} минут. "
        f"На ручную проверку (аудио сохранено): {len(rejected)}. "
        f"Возвращено естественных фрагментов без точки: {recovered_clauses}. "
        f"Длительность: {df['duration'].min():.1f}–{df['duration'].max():.1f} сек, "
        f"медиана {df['duration'].median():.1f} сек. "
        f"Громкость нормализована около -20 dBFS. "
        f"Датасет: {drive_dataset}"
    )
    editor_text = "\n".join(f"{row['file']}|{row['text']}" for row in rows)
    return df, summary, str(review_csv), editor_text

def create_verified_project(source_project, target_project, reviewed_metadata_file, recovered_archive=None):
    """Build a separate dataset from listener-approved WAVs; never run ASR on them."""
    import csv
    import io
    if not reviewed_metadata_file:
        raise ValueError("Загрузите metadata_checked.csv после прослушивания фраз.")
    source_name, target_name = safe_name(source_project), safe_name(target_project)
    if source_name == target_name:
        raise ValueError("Нужен НОВЫЙ проект, иначе исходный датасет будет затронут.")
    _, source_drive, _ = project_paths(source_name)
    source_dataset = source_drive / "dataset"
    source_review = source_dataset / "review.csv"
    source_wav = source_dataset / "wav"
    if not source_review.exists() or not source_wav.exists():
        raise FileNotFoundError("Исходный датасет на Google Drive не найден.")
    local_target, target_drive, _ = project_paths(target_name)
    target_dataset = local_target / "dataset"
    drive_dataset = target_drive / "dataset"
    if (target_dataset / "metadata.csv").exists() or drive_dataset.exists():
        raise FileExistsError("Проект уже содержит датасет. Укажите другое название.")
    original_review = pd.read_csv(source_review, dtype={"file": str})
    if original_review["file"].duplicated().any():
        raise ValueError("В исходном review.csv повторяются имена WAV.")
    known_original = set(original_review["file"])
    approved = []
    seen = set()
    for line_num, raw in enumerate(Path(reviewed_metadata_file).read_text(encoding="utf-8-sig").splitlines(), 1):
        if not raw.strip():
            continue
        if raw.count("|") != 1:
            raise ValueError(f"Строка {line_num}: требуется имя.wav|точная фраза.")
        name, text = raw.split("|", 1)
        name, text = name.strip(), normalize_text(text)
        if not name or not text or name in seen or Path(name).name != name or not name.endswith(".wav"):
            raise ValueError(f"Строка {line_num}: пустой текст или неверное/повторное имя WAV.")
        seen.add(name)
        approved.append((name, text))
    if not approved:
        raise ValueError("Нет ни одной подтверждённой фразы.")
    needs_recovered = {name for name, _ in approved if name not in known_original}
    recovered_rows = {}
    recovered_members = {}
    if needs_recovered:
        if not recovered_archive:
            raise ValueError("Среди подтверждённых есть восстановленные WAV. Загрузите recovered_audio_for_colab.zip.")
        with zipfile.ZipFile(recovered_archive) as archive:
            infos = archive.infolist()
            if len(infos) > 300 or sum(info.file_size for info in infos) > 250 * 1024 * 1024:
                raise ValueError("Слишком большой архив восстановленных фрагментов.")
            members = set(archive.namelist())
            if "recovered_review.csv" not in members:
                raise ValueError("В архиве нет recovered_review.csv.")
            content = archive.read("recovered_review.csv").decode("utf-8-sig")
            reader = csv.DictReader(io.StringIO(content))
            required = {"file", "start_sec", "end_sec", "duration", "text"}
            if not reader.fieldnames or not required.issubset(reader.fieldnames):
                raise ValueError("Неверный формат recovered_review.csv.")
            for row in reader:
                name = row["file"]
                member = f"recovered_audio/{name}"
                if (
                    not name.startswith("my_voice_recovered_") or
                    Path(name).name != name or
                    not name.endswith(".wav") or
                    name in recovered_rows or
                    member not in members
                ):
                    raise ValueError(f"Неверная запись восстановленного WAV: {name}")
                start, end = float(row["start_sec"]), float(row["end_sec"])
                if not 0 <= start < end or end - start > 30:
                    raise ValueError(f"Неверные временные границы для {name}.")
                recovered_rows[name] = row
                recovered_members[name] = member
            unknown = needs_recovered - set(recovered_rows)
            if unknown:
                raise ValueError(f"Неподтверждённый или отсутствующий WAV: {sorted(unknown)[0]}")
    candidate = local_target / "dataset_verified_candidate"
    if candidate.exists():
        shutil.rmtree(candidate)
    (candidate / "wav").mkdir(parents=True)
    try:
        records = []
        archive = zipfile.ZipFile(recovered_archive) if needs_recovered else None
        try:
            originals = original_review.set_index("file")
            for name, text in approved:
                dest = candidate / "wav" / name
                if name in known_original:
                    if not (source_wav / name).is_file():
                        raise FileNotFoundError(f"Исходное аудио отсутствует: {name}")
                    shutil.copy2(source_wav / name, dest)
                    record = originals.loc[name].to_dict()
                else:
                    info = archive.getinfo(recovered_members[name])
                    if info.file_size > 30 * 22050 * 2 + 4096:
                        raise ValueError(f"Восстановленный WAV слишком большой: {name}")
                    with archive.open(info) as src, dest.open("wb") as dst:
                        shutil.copyfileobj(src, dst)
                    record = recovered_rows[name].copy()
                record["file"], record["text"] = name, text
                records.append(record)
        finally:
            if archive is not None:
                archive.close()
        (candidate / "metadata.csv").write_text(
            "\n".join(name + "|" + text for name, text in approved) + "\n",
            encoding="utf-8",
        )
        pd.DataFrame(records).to_csv(candidate / "review.csv", index=False, encoding="utf-8")
        count, minutes = validate_dataset_for_training(candidate)
        (candidate / "approved_by_review.txt").write_text(
            f"Подтверждено вручную: {count} фраз; {minutes:.2f} минуты.\n"
            f"Источник: {source_name}. Дополнительных восстановленных WAV: {len(needs_recovered)}.\n",
            encoding="utf-8",
        )
        shutil.copytree(candidate, drive_dataset)
        if target_dataset.exists():
            shutil.rmtree(target_dataset)
        shutil.move(str(candidate), str(target_dataset))
    except Exception:
        if candidate.exists():
            shutil.rmtree(candidate)
        raise
    return (
        f"Создан отдельный проект {target_name}: {count} подтверждённых фраз, "
        f"{minutes:.1f} минуты. Из восстановленных: {len(needs_recovered)}. "
        f"Датасет на Google Drive: {drive_dataset}. Обучение НЕ запущено.",
        target_name,
    )


def apply_text_review(project, editor_text):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)
    dataset_dir = local / "dataset"
    wav_dir = dataset_dir / "wav"

    clean_rows = []
    errors = []
    for line_no, raw in enumerate(str(editor_text).splitlines(), start=1):
        raw = raw.strip()
        if not raw:
            continue
        if "|" not in raw:
            errors.append(f"строка {line_no}: нет символа |")
            continue
        filename, text = raw.split("|", 1)
        filename = filename.strip()
        text = normalize_text(text)
        if not filename or not text:
            errors.append(f"строка {line_no}: пустое имя файла или текст")
            continue
        if Path(filename).name != filename or not (wav_dir / filename).exists():
            errors.append(f"строка {line_no}: аудиофайл {filename} не найден")
            continue
        clean_rows.append((filename, text))

    if errors:
        raise ValueError("Исправьте ошибки:\n" + "\n".join(errors[:20]))
    if not clean_rows:
        raise ValueError("Нет ни одной строки вида имя.wav|текст")

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for filename, text in clean_rows:
            f.write(f"{filename}|{text}\n")

    drive_dataset = drive_p / "dataset"
    drive_dataset.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata, drive_dataset / "metadata.csv")
    return (
        f"Сохранено {len(clean_rows)} фраз. "
        f"metadata.csv обновлён и скопирован на Google Drive."
    )

def apply_review(project, review_file):
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)
    dataset_dir = local / "dataset"
    review_csv = Path(review_file) if review_file else dataset_dir / "review.csv"
    df = pd.read_csv(review_csv)

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for _, row in df.iterrows():
            text = normalize_text(row["text"])
            if text:
                f.write(f"{row['file']}|{text}\n")

    drive_dataset = drive_p / "dataset"
    drive_dataset.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata, drive_dataset / "metadata.csv")
    shutil.copy2(review_csv, drive_dataset / "review.csv")
    return f"Исправления применены. metadata.csv обновлён: {metadata}"


In [ ]:
#@title 4. Backend: обучение, checkpoints и экспорт
def restore_dataset_from_drive(project):
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)
    src = drive_p / "dataset"
    dst = local / "dataset"
    if not src.exists():
        raise FileNotFoundError(f"На Google Drive нет датасета для проекта {project}")
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return dst

def newest_checkpoint(path: Path):
    files = [p for p in path.glob("*.ckpt") if p.name != "last.ckpt"]
    return max(files, key=lambda p: p.stat().st_mtime) if files else None

def sync_checkpoints(local_ckpt: Path, drive_ckpt: Path, keep_last: int):
    drive_ckpt.mkdir(parents=True, exist_ok=True)
    for src in local_ckpt.glob("*.ckpt"):
        dst = drive_ckpt / src.name
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            try:
                shutil.copy2(src, dst)
            except Exception:
                pass

    if keep_last > 0:
        for folder in (drive_ckpt, local_ckpt):
            ckpts = sorted(
                folder.glob("*.ckpt"),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            for old in ckpts[int(keep_last):]:
                old.unlink(missing_ok=True)

def checkpoint_epoch(path):
    try:
        data = torch.load(str(path), map_location="cpu", weights_only=False)
        epoch = int(data.get("epoch", -1))
        del data
        gc.collect()
        return epoch
    except Exception as e:
        print("Не удалось прочитать номер эпохи checkpoint:", e)
        return -1


def resolve_start_checkpoint(project, start_mode):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)

    if start_mode == "Последний checkpoint с Google Drive":
        candidates = []

        local_full = newest_checkpoint(local / "checkpoints")
        if local_full:
            candidates.append((local_full, "resume"))

        drive_full = newest_checkpoint(drive_p / "checkpoints")
        if drive_full:
            candidates.append((drive_full, "resume"))

        drive_recovery = newest_checkpoint(drive_p / "recovery")
        if drive_recovery:
            candidates.append((drive_recovery, "recovery"))

        if candidates:
            path, kind = max(
                candidates,
                key=lambda item: item[0].stat().st_mtime,
            )
            return str(path), kind

    return str(DMITRI_CKPT), "base"


def train_voice(
    project,
    additional_epochs,
    batch_size,
    save_mode,
    save_every,
    keep_last,
    drive_backup_every,
    start_mode,
):
    global _train_process
    project = safe_name(project)
    unload_asr()

    local, drive_p, _ = project_paths(project)
    dataset = local / "dataset"
    drive_dataset = drive_p / "dataset"
    drive_metadata = drive_dataset / "metadata.csv"
    local_metadata = dataset / "metadata.csv"
    if _train_process is not None and _train_process.poll() is None:
        raise RuntimeError("Обучение уже запущено. Сначала остановите текущий процесс.")
    # Drive is the canonical copy: the user may have corrected metadata.csv
    # after the dataset was prepared, while Colab still has an older cache.
    # Also refresh if local WAV files are missing or belong to an older set.
    must_restore = not local_metadata.is_file()
    if drive_metadata.is_file() and not must_restore:
        must_restore = local_metadata.read_bytes() != drive_metadata.read_bytes()
        if not must_restore:
            wav_dir_local = dataset / "wav"
            names = [
                line.split("|", 1)[0].strip()
                for line in drive_metadata.read_text(encoding="utf-8").splitlines()
                if line.strip()
            ]
            must_restore = any(not (wav_dir_local / name).is_file() for name in names)
    if must_restore:
        restore_dataset_from_drive(project)
        # Previously phonemized labels must not survive text corrections.
        cache_old = local / "cache"
        if cache_old.exists():
            shutil.rmtree(cache_old)

    wav_dir = dataset / "wav"
    metadata = dataset / "metadata.csv"
    checked_phrases, checked_minutes = validate_dataset_for_training(dataset)
    cache_dir = local / "cache"
    config_path = drive_p / f"ru_RU-{project}-medium.onnx.json"
    local_ckpt = local / "checkpoints"
    drive_recovery = drive_p / "recovery"

    cache_dir.mkdir(parents=True, exist_ok=True)
    local_ckpt.mkdir(parents=True, exist_ok=True)
    drive_recovery.mkdir(parents=True, exist_ok=True)

    start_ckpt, start_kind = resolve_start_checkpoint(
        project,
        start_mode,
    )
    is_resume = start_kind == "resume"
    start_epoch = checkpoint_epoch(start_ckpt)
    additional_epochs = max(1, int(additional_epochs))

    target_max_epochs = (
        start_epoch + 1 + additional_epochs
        if is_resume and start_epoch >= 0
        else additional_epochs
    )

    save_every = max(1, int(save_every))
    keep_last = max(1, int(keep_last))
    drive_backup_every = max(1, int(drive_backup_every))

    runner = local / "run_piper_training.py"
    if save_mode == "Каждые N эпох":
        schedule_arg = f"every_n_epochs={save_every}"
    else:
        schedule_arg = f"every_n_train_steps={save_every}"

    runner.write_text(
        f"""
import shutil
from pathlib import Path

import torch
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
import piper.train.__main__ as piper_train


class PruningModelCheckpoint(ModelCheckpoint):
    def _save_checkpoint(self, trainer, filepath):
        super()._save_checkpoint(trainer, filepath)
        root = Path(self.dirpath)
        keep = {keep_last}
        named = sorted(
            root.glob("*.ckpt"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        for old in named[keep:]:
            try:
                old.unlink()
            except FileNotFoundError:
                pass


class DriveRecoveryCallback(Callback):
    def __init__(self, drive_dir, temp_file, interval):
        super().__init__()
        self.drive_dir = Path(drive_dir)
        self.temp_file = Path(temp_file)
        self.interval = max(1, int(interval))
        self.last_saved_epoch = None
        self.drive_dir.mkdir(parents=True, exist_ok=True)

    def _save_recovery(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        step = int(trainer.global_step)

        state_dict = {{
            key: value.detach().cpu()
            for key, value in pl_module.state_dict().items()
        }}
        payload = {{
            "state_dict": state_dict,
            "epoch": epoch,
            "global_step": step,
            "piper_recovery_weights_only": True,
        }}

        torch.save(payload, self.temp_file)
        target = self.drive_dir / (
            f"recovery-{{epoch:04d}}-{{step}}.ckpt"
        )
        shutil.copy2(self.temp_file, target)
        self.temp_file.unlink(missing_ok=True)
        self.last_saved_epoch = epoch
        print(
            f"[Drive recovery] saved {{target.name}} "
            f"({{target.stat().st_size / 1024 / 1024:.1f}} MiB)"
        )

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        if (epoch + 1) % self.interval == 0:
            self._save_recovery(trainer, pl_module)

    def on_train_end(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        if self.last_saved_epoch != epoch:
            self._save_recovery(trainer, pl_module)


checkpoint = PruningModelCheckpoint(
    dirpath={str(local_ckpt)!r},
    save_top_k=-1,
    save_last=False,
    filename={project!r} + "-{{epoch:04d}}-{{step}}",
    auto_insert_metric_name=False,
    {schedule_arg},
)

recovery = DriveRecoveryCallback(
    drive_dir={str(drive_recovery)!r},
    temp_file={str(local / "recovery-tmp.ckpt")!r},
    interval={drive_backup_every},
)

piper_train._DEFAULT_CALLBACKS = [checkpoint, recovery]
piper_train.main()
""".strip() + "\n",
        encoding="utf-8",
    )

    cmd = [
        sys.executable,
        str(runner),
        "fit",
        "--data.voice_name",
        project,
        "--data.csv_path",
        str(metadata),
        "--data.audio_dir",
        str(wav_dir),
        "--model.sample_rate",
        "22050",
        "--data.espeak_voice",
        "ru",
        "--data.cache_dir",
        str(cache_dir),
        "--data.config_path",
        str(config_path),
        "--data.batch_size",
        str(int(batch_size)),
    ]

    if is_resume:
        cmd += ["--ckpt_path", start_ckpt]
    else:
        cmd += ["--model.warmstart_ckpt", start_ckpt]

    cmd += [
        "--trainer.max_epochs",
        str(target_max_epochs),
        "--trainer.accelerator",
        "gpu",
        "--trainer.devices",
        "1",
        "--trainer.precision",
        "16-mixed",
        "--trainer.log_every_n_steps",
        "1",
        "--trainer.enable_checkpointing",
        "true",
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

    if start_kind == "resume":
        start_label = "точный resume полного checkpoint"
    elif start_kind == "recovery":
        start_label = "warm-start recovery с Drive (без optimizer)"
    else:
        start_label = "warm-start от Dmitri"

    log_file = drive_p / "training.log"
    header = (
        "Старт обучения\n"
        f"Проект: {project}\n"
        f"Checkpoint: {start_ckpt}\n"
        f"Режим старта: {start_label}\n"
        f"Эпоха источника: {start_epoch if start_epoch >= 0 else 'неизвестна'}\n"
        f"Проверено реальных фраз: {checked_phrases} ({checked_minutes:.1f} мин)\n"
        f"Дополнительных эпох: {additional_epochs}\n"
        f"Целевая max_epochs Lightning: {target_max_epochs}\n"
        f"Полные checkpoints локально: {local_ckpt}\n"
        f"Recovery на Drive каждые {drive_backup_every} эпох: {drive_recovery}\n\n"
    )
    yield header

    _train_process = subprocess.Popen(
        cmd,
        cwd="/content/piper1-gpl",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    collected = [header]
    with log_file.open("a", encoding="utf-8") as lf:
        for line in _train_process.stdout:
            collected.append(line)
            lf.write(line)
            lf.flush()
            yield "".join(collected[-220:])

    code = _train_process.wait()
    _train_process = None

    if code == 0:
        yield (
            "".join(collected[-220:])
            + "\n\nОбучение завершено. "
            "Полные checkpoints остались локально, recovery сохранены на Google Drive."
        )
    else:
        yield (
            "".join(collected[-220:])
            + f"\n\nОбучение остановилось с кодом {code}."
        )


def stop_training():
    global _train_process
    if _train_process is not None and _train_process.poll() is None:
        _train_process.terminate()
        return "Запрошена остановка обучения. Текущие сохранённые checkpoints останутся."
    return "Активного обучения сейчас нет."

def export_voice(project):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)
    drive_ckpt = drive_p / "checkpoints"
    ckpt = newest_checkpoint(local / "checkpoints")
    if ckpt is None:
        ckpt = newest_checkpoint(drive_ckpt)
    if ckpt is None:
        raise FileNotFoundError("Checkpoint не найден.")

    config = local / f"ru_RU-{project}-medium.onnx.json"
    if not config.exists():
        drive_config = drive_p / f"ru_RU-{project}-medium.onnx.json"
        if drive_config.exists():
            shutil.copy2(drive_config, config)
        else:
            raise FileNotFoundError("Не найден config JSON, созданный Piper во время обучения.")

    out_dir = local / "export"
    out_dir.mkdir(parents=True, exist_ok=True)
    model_name = f"ru_RU-{project}-medium"
    onnx = out_dir / f"{model_name}.onnx"
    json_out = out_dir / f"{model_name}.onnx.json"

    export_env = os.environ.copy()
    export_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    subprocess.run(
        [
            sys.executable, "-m", "piper.train.export_onnx",
            "--checkpoint", str(ckpt),
            "--output-file", str(onnx),
        ],
        cwd="/content/piper1-gpl",
        env=export_env,
        check=True,
    )
    shutil.copy2(config, json_out)

    zip_path = out_dir / f"{model_name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(onnx, onnx.name)
        z.write(json_out, json_out.name)

    drive_export = drive_p / "export"
    drive_export.mkdir(parents=True, exist_ok=True)
    shutil.copy2(onnx, drive_export / onnx.name)
    shutil.copy2(json_out, drive_export / json_out.name)
    shutil.copy2(zip_path, drive_export / zip_path.name)

    return str(zip_path), f"Экспорт готов: {drive_export / zip_path.name}"


In [ ]:
#@title 5. Запуск веб-интерфейса
import gradio as gr

with gr.Blocks(title="Piper Trainer RU") as demo:
    gr.Markdown(
        "# Piper Trainer RU\n"
        "Qwen3-ASR 1.7B (вся запись длинными блоками) → ForcedAligner 0.6B (тайм-коды слов) → WAV по 3–6 сек → Piper → Google Drive"
    )

    with gr.Row():
        env_btn = gr.Button("Проверить окружение")
        env_status = gr.Textbox(label="Окружение", lines=7)
    env_btn.click(environment_status, outputs=env_status)
    demo.load(environment_status, outputs=env_status)

    with gr.Tab("1. Датасет"):
        project = gr.Textbox(
            label="Название проекта",
            value="my_voice",
            info="Латиница/кириллица допустимы. Пробелы будут заменены на подчёркивания."
        )
        files = gr.Files(
            label="Длинные записи или ZIP с аудиофайлами",
            file_types=["audio", ".zip"],
            type="filepath",
        )
        drive_audio_path = gr.Textbox(
            label="Или путь к исходной записи на Google Drive",
            placeholder="/content/drive/MyDrive/PiperTrainer/my_voice/reconstructed_source.wav",
            info="Можно не загружать файл заново: вставьте путь к уже сохранённой записи на Drive.",
        )

        with gr.Row():
            min_silence = gr.Slider(80, 800, value=250, step=10, label="Пауза для разреза, мс")
            silence_db = gr.Slider(-60, -20, value=-35, step=1, label="Порог тишины, дБ")
        with gr.Row():
            min_sec = gr.Slider(0.5, 5.0, value=3.0, step=0.1, label="Минимум фразы, сек (цель: 3–6)")
            max_sec = gr.Slider(3, 20, value=6, step=1, label="Максимум фразы, сек (цель: 3–6)")
            padding = gr.Slider(0, 800, value=90, step=10, label="Запас у границ слов, мс")

        with gr.Row():
            asr_test_file = gr.File(
                label="Одно аудио для проверки Qwen",
                file_types=["audio"],
                type="filepath",
            )
            asr_test_btn = gr.Button("Проверить Qwen на одном аудиофайле")
        asr_test_result = gr.Textbox(label="Результат проверки Qwen", lines=4)

        prepare_btn = gr.Button("Распознать и подготовить датасет", variant="primary")
        dataset_status = gr.Textbox(label="Статус", lines=3)
        table = gr.Dataframe(
            headers=["file", "text", "duration", "source", "start_sec", "end_sec"],
            label="Результат. Текст можно исправить прямо в таблице.",
            interactive=True,
        )
        review_file = gr.File(label="review.csv")
        metadata_editor = gr.Textbox(
            label="Текстовый редактор для VoiceOver: имя.wav|текст",
            lines=18,
            info=(
                "Одна фраза на строку. Можно исправлять только текст после первого символа |. "
                "Этот редактор проще таблицы для VoiceOver."
            ),
        )
        with gr.Row():
            apply_text_btn = gr.Button("Сохранить текстовый редактор", variant="primary")
            apply_btn = gr.Button("Применить исправления из таблицы")

        def save_table(project_name, data):
            p = safe_name(project_name)
            local, drive_p, _ = project_paths(p)
            dataset = local / "dataset"
            df = pd.DataFrame(data)
            path = dataset / "review.csv"
            df.to_csv(path, index=False, encoding="utf-8")
            return apply_review(p, str(path))

        asr_test_btn.click(
            test_qwen_asr,
            inputs=asr_test_file,
            outputs=asr_test_result,
        )
        prepare_btn.click(
            prepare_dataset,
            inputs=[files, project, min_silence, silence_db, min_sec, max_sec, padding, drive_audio_path],
            outputs=[table, dataset_status, review_file, metadata_editor],
        )
        apply_text_btn.click(
            apply_text_review,
            inputs=[project, metadata_editor],
            outputs=dataset_status,
        )
        apply_btn.click(save_table, inputs=[project, table], outputs=dataset_status)

        gr.Markdown(
            "### Отдельный проект из проверенных записей\n"
            "Прослушайте WAV и исправьте текст в локальной странице проверки. "
            "Загрузите полученный metadata_checked.csv. Старый проект останется нетронутым."
        )
        checked_file = gr.File(
            label="metadata_checked.csv с подтверждёнными фразами",
            file_types=[".csv"], type="filepath",
        )
        checked_name = gr.Textbox(
            label="Название НОВОГО проверенного проекта",
            value="my_voice_verified",
        )
        recovered_file = gr.File(
            label="ZIP восстановленных WAV (если вы подтвердили такие фразы)",
            file_types=[".zip"],
            type="filepath",
        )
        checked_button = gr.Button("Создать новый проект из проверенных фраз")
        checked_result = gr.Textbox(label="Результат проверки", lines=4)
        checked_button.click(
            create_verified_project,
            inputs=[project, checked_name, checked_file, recovered_file],
            outputs=[checked_result, project],
        )

    with gr.Tab("2. Обучение"):
        gr.Markdown("Перед обучением Qwen ASR выгружается из GPU. Полные checkpoints хранятся локально, а компактные recovery-копии периодически сохраняются на Google Drive.")
        with gr.Row():
            epochs = gr.Number(
                value=500,
                precision=0,
                label="Сколько дополнительных эпох обучить",
                info="Считается поверх эпохи выбранного checkpoint. Для первого теста можно поставить 10–20.",
            )
            batch = gr.Dropdown([4, 6, 8, 12, 16], value=8, label="Batch size")
        with gr.Row():
            save_mode = gr.Radio(
                ["Каждые N эпох", "Каждые N шагов"],
                value="Каждые N эпох",
                label="Когда сохранять checkpoint",
            )
            save_every = gr.Number(value=5, precision=0, label="N")
            keep_last = gr.Number(
                value=3,
                precision=0,
                label="Сколько последних полных checkpoints хранить локально",
            )
            drive_backup_every = gr.Number(
                value=100,
                precision=0,
                label="Recovery на Google Drive каждые N эпох",
                info="Recovery весит примерно в 3 раза меньше полного checkpoint и нужен для восстановления после отключения Colab.",
            )
        start_mode = gr.Radio(
            ["Dmitri medium (база)", "Последний checkpoint с Google Drive"],
            value="Dmitri medium (база)",
            label="Откуда продолжать",
        )

        with gr.Row():
            train_btn = gr.Button("Начать обучение", variant="primary")
            stop_btn = gr.Button("Остановить")
        train_log = gr.Textbox(label="Лог обучения", lines=22, autoscroll=True)

        train_btn.click(
            train_voice,
            inputs=[
                project,
                epochs,
                batch,
                save_mode,
                save_every,
                keep_last,
                drive_backup_every,
                start_mode,
            ],
            outputs=train_log,
        )
        stop_btn.click(stop_training, outputs=train_log)

    with gr.Tab("3. Экспорт"):
        export_btn = gr.Button("Экспортировать последний checkpoint в ONNX", variant="primary")
        exported_zip = gr.File(label="ZIP для Piper Voice")
        export_status = gr.Textbox(label="Статус")
        export_btn.click(export_voice, inputs=[project], outputs=[exported_zip, export_status])

    with gr.Tab("Справка"):
        gr.Markdown(
            "### Рекомендуемые настройки для T4\n"
            "- Batch size: 8. Если VRAM заканчивается — 4 или 6.\n"
            "- Для smoke test: 10–20 дополнительных эпох. Для нормального обучения — больше.\n"
            "- Checkpoint: каждые 5–10 эпох.\n"
            "- Хранить: 3 последних, потому что каждый checkpoint большой.\n"
            "- Для первого опыта достаточно 30–60 минут чистой речи, но больше обычно лучше.\n\n"
            "### Google Drive\n"
            "Всё лежит в `MyDrive/PiperTrainer/<проект>/`."
        )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)
